# 01 — Dataset inventory

This notebook performs the first structural inspection of the raw datasets used in the pan-cancer epigenetic resistance project.

The goal is to verify:

- file availability
- file size
- table dimensions
- column structure
- identifier columns
- missingness
- duplicated keys
- basic compatibility between DepMap and GDSC inputs

Raw data are treated as read-only.

In [1]:
from pathlib import Path
import sys

import pandas as pd

from src.utils.paths import Paths, project_relative_path, find_repo_root

In [2]:
# Define project root and data directories

PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT = Paths.root
DEPMAP_DIR = Paths.depmap
GDSC_DIR = Paths.gdsc

FILES = {
    "depmap_model": DEPMAP_DIR / "Model.csv",
    "depmap_omics_profiles": DEPMAP_DIR / "OmicsProfiles.csv",
    "depmap_expression": DEPMAP_DIR / "OmicsExpressionProteinCodingGenesTPMLogp1.csv",
    "depmap_mutations": DEPMAP_DIR / "OmicsSomaticMutations.csv",
    "gdsc_ic50": GDSC_DIR / "GDSC2_fitted_dose_response_27Oct23.xlsx",
}

print(
    f"Project root: "
    f"{project_relative_path(PROJECT_ROOT)}"
)


Project root: .


In [3]:
# Check if files exist and print their sizes
for name, path in FILES.items():
    size_mb = path.stat().st_size / 1e6 if path.exists() else None
    print(f"{name:25} | exists={path.exists()} | size_mb={size_mb}")

depmap_model              | exists=True | size_mb=0.645696
depmap_omics_profiles     | exists=True | size_mb=0.254733
depmap_expression         | exists=True | size_mb=506.628654
depmap_mutations          | exists=True | size_mb=338.945382
gdsc_ic50                 | exists=True | size_mb=22.112537


In [4]:
# Preview the first few rows of each file and collect inventory information
inventory_rows = []

for name, path in FILES.items():

    if path.suffix == ".csv":
        df = pd.read_csv(path, nrows=5)

    elif path.suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path, nrows=5)

    else:
        continue

    inventory_rows.append({
        "dataset": name,
        "rows_preview": df.shape[0],
        "columns": df.shape[1],
        "column_names_preview": list(df.columns[:10]),
    })

inventory_df = pd.DataFrame(inventory_rows)

inventory_df

,dataset,rows_preview,columns,column_names_preview
0,depmap_model,5,47,"[ModelID, PatientID, CellLineName, StrippedCel..."
1,depmap_omics_profiles,5,9,"[ProfileID, ModelCondition, ModelID, Datatype,..."
2,depmap_expression,5,19194,"[Unnamed: 0, TSPAN6 (7105), TNMD (64102), DPM1..."
3,depmap_mutations,5,70,"[Chrom, Pos, Ref, Alt, AF, DP, RefCount, AltCo..."
4,gdsc_ic50,5,16,"[DATASET, NLME_RESULT_ID, NLME_CURVE_ID, CELL_..."


In [5]:
# Print column names for each file, handling cases with many columns
for name, path in FILES.items():
    print("=" * 80)
    print(name)
    
    if path.suffix == ".csv":
        df_preview = pd.read_csv(path, nrows=5)
    else:
        df_preview = pd.read_excel(path, nrows=5)
    
    print(f"Columns: {df_preview.shape[1]}")
    
    if df_preview.shape[1] <= 80:
        print(list(df_preview.columns))
    else:
        print("First 15 columns:")
        print(list(df_preview.columns[:15]))
        print("Last 15 columns:")
        print(list(df_preview.columns[-15:]))

depmap_model
Columns: 47
['ModelID', 'PatientID', 'CellLineName', 'StrippedCellLineName', 'DepmapModelType', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'OncotreeCode', 'PatientSubtypeFeatures', 'RRID', 'Age', 'AgeCategory', 'Sex', 'PatientRace', 'PrimaryOrMetastasis', 'SampleCollectionSite', 'SourceType', 'SourceDetail', 'CatalogNumber', 'ModelType', 'TissueOrigin', 'ModelDerivationMaterial', 'ModelTreatment', 'PatientTreatmentStatus', 'PatientTreatmentType', 'PatientTreatmentDetails', 'Stage', 'StagingSystem', 'PatientTumorGrade', 'PatientTreatmentResponse', 'GrowthPattern', 'OnboardedMedia', 'FormulationID', 'SerumFreeMedia', 'PlateCoating', 'EngineeredModel', 'EngineeredModelDetails', 'CulturedResistanceDrug', 'PublicComments', 'CCLEName', 'HCMIID', 'ModelAvailableInDbgap', 'ModelSubtypeFeatures', 'WTSIMasterCellID', 'SangerModelID', 'COSMICID']
depmap_omics_profiles
Columns: 9
['ProfileID', 'ModelCondition', 'ModelID', 'Datatype', 'WESKit', 'Product', 'Stranded

In [6]:
# Check for key columns and their properties
key_checks = []

checks = {
    "depmap_model": ("csv", FILES["depmap_model"], ["ModelID", "SangerModelID", "COSMICID", "CellLineName"]),
    "depmap_omics_profiles": ("csv", FILES["depmap_omics_profiles"], ["ModelID", "ProfileID", "SequencingID"]),
    "depmap_expression": ("csv", FILES["depmap_expression"], ["ModelID", "SequencingID"]),
    "depmap_mutations": ("csv", FILES["depmap_mutations"], ["ModelID", "SequencingID", "HugoSymbol"]),
    "gdsc_ic50": ("excel", FILES["gdsc_ic50"], ["SANGER_MODEL_ID", "COSMIC_ID", "CELL_LINE_NAME", "DRUG_NAME"]),
}

for dataset, (kind, path, cols) in checks.items():
    if kind == "csv":
        df = pd.read_csv(path, usecols=lambda c: c in cols)
    else:
        df = pd.read_excel(path, usecols=lambda c: c in cols)

    for col in cols:
        if col in df.columns:
            key_checks.append({
                "dataset": dataset,
                "column": col,
                "rows": len(df),
                "missing": df[col].isna().sum(),
                "unique": df[col].nunique(dropna=True),
                "duplicated_values": df[col].duplicated().sum(),
            })

key_checks_df = pd.DataFrame(key_checks)
key_checks_df

,dataset,column,rows,missing,unique,duplicated_values
0,depmap_model,ModelID,2105,0,2105,0
1,depmap_model,SangerModelID,2105,887,1217,887
2,depmap_model,COSMICID,2105,1128,977,1127
3,depmap_model,CellLineName,2105,0,2105,0
4,depmap_omics_profiles,ModelID,4518,0,2005,2513
5,depmap_omics_profiles,ProfileID,4518,0,4518,0
6,depmap_mutations,ModelID,718369,0,1929,716440
7,depmap_mutations,HugoSymbol,718369,0,19584,698785
8,gdsc_ic50,SANGER_MODEL_ID,242036,0,969,241067
9,gdsc_ic50,CELL_LINE_NAME,242036,0,969,241067


In [7]:
# Focused exploration of the Omics Profiles dataset
omics_profiles = pd.read_csv(FILES["depmap_omics_profiles"])

omics_profiles[
    [
        "ModelID",
        "ModelCondition",
        "ProfileID",
        "Datatype",
        "Source",
    ]
].head(10)

,ModelID,ModelCondition,ProfileID,Datatype,Source
0,ACH-000957,MC-000957-Yckn,PR-01r7OM,rna,BROAD
1,ACH-002785,MC-002785-qo9e,PR-02XmLG,rna,BROAD
2,ACH-003273,MC-003273-qwen,PR-045poV,rna,BROAD
3,ACH-001289,MC-001289-BpdI,PR-04VvBz,wes,BROAD
4,ACH-000237,MC-000237-Fv8V,PR-08baLC,wgs,BROAD
5,ACH-000520,MC-000520-YIm7,PR-09gmEI,rna,BROAD
6,ACH-002269,MC-002269-tG4R,PR-09S1KU,wes,SANGER
7,ACH-000575,MC-000575-q3Ch,PR-0B6q0Z,rna,BROAD
8,ACH-000074,MC-000074-Qybw,PR-0c2Lny,wes,SANGER
9,ACH-000869,MC-000869-u5uo,PR-0ccWFQ,rna,BROAD


In [8]:
# Summarize the Omics Profiles dataset by data type

omics_profiles = pd.read_csv(FILES["depmap_omics_profiles"])

summary = (
    omics_profiles
    .groupby("Datatype")
    .agg(
        rows=("ProfileID", "size"),
        unique_profiles=("ProfileID", "nunique"),
        unique_models=("ModelID", "nunique"),
        unique_conditions=("ModelCondition", "nunique"),
    )
    .reset_index()
)

summary.sort_values("rows", ascending=False)

,Datatype,rows,unique_profiles,unique_models,unique_conditions
1,wes,1909,1909,1700,1909
0,rna,1690,1690,1673,1674
2,wgs,919,919,917,917


In [9]:
# Source distribution

omics_profiles["Source"].value_counts()

Source
BROAD     3492
SANGER    1026
Name: count, dtype: int64

In [10]:
# Product distribution

omics_profiles["Product"].value_counts(dropna=False)

Product
NaN              4056
NOVA_SEQ_6000     412
HISEQ_2500         22
HISEQ_4000         21
HISEQ_X_10          7
Name: count, dtype: int64

In [11]:
# Explore the overlap of Sanger Model IDs between DepMap and GDSC datasets
model_df = pd.read_csv(FILES["depmap_model"])

gdsc_df = pd.read_excel(FILES["gdsc_ic50"])

depmap_sanger_ids = (
    model_df["SangerModelID"]
    .dropna()
    .astype(str)
    .unique()
)

gdsc_sanger_ids = (
    gdsc_df["SANGER_MODEL_ID"]
    .dropna()
    .astype(str)
    .unique()
)

shared_ids = set(depmap_sanger_ids) & set(gdsc_sanger_ids)

print(f"DepMap Sanger IDs : {len(depmap_sanger_ids)}")
print(f"GDSC Sanger IDs   : {len(gdsc_sanger_ids)}")
print(f"Shared IDs        : {len(shared_ids)}")

DepMap Sanger IDs : 1217
GDSC Sanger IDs   : 969
Shared IDs        : 967
